Instalar optuna no Colab (único módulo dos que não usarei que não está pré instalado)

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 13.2 MB/s eta 0:00:00


Importando módulos

In [2]:
from google.colab import drive
import os
import glob
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

import optuna

Definindo o caminho (remoto) dos dados

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ROOT = '/content/drive/MyDrive/colab_data/rna1/lista5/Treino'

Definindo questões operacionais: Modo fast debug para rodar rapidinho e verificar erros e problemas, desligar para rodar de verdade

fixando o tamanho da imagem para upscale - como estou usando modelos pré treinados para 224x224, a recomendação da bibliografia é (neste caso) aumentar o tamanho das imagens de 96x103 para 224x224, para garantir o correto funcionamento dos filtros kernel de imagem da forma que foram concebidos nos modelos originais.

In [5]:
FAST_DEBUG = False
if FAST_DEBUG:
    N_EPOCHS = 2
    N_TRIALS = 2
    BATCH_SIZE = 16
else:
    N_EPOCHS = 20
    N_TRIALS = 30
    BATCH_SIZE = 32

IMAGE_HEIGHT = 224
IMAGE_WIDTH  = 224

Garantindo possibilidades: rodar na GPU se possível, no processador, caso não tenha.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)
print("FAST_DEBUG:", FAST_DEBUG)

DEVICE: cuda
FAST_DEBUG: False


Trazer os dados para o python

In [7]:
def collect_image_files(root):
    exts = ("*.bmp", "*.BMP")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(root, e)))
    files = sorted(files)
    return files

all_files = collect_image_files(ROOT)
if len(all_files) == 0:
    raise RuntimeError(f"Nenhuma imagem encontrada")

Preparando os dados pré-separação em treino e validação


In [8]:
class FingerprintDataset(Dataset):
    def __init__(self, files_list, transform=None):
        self.files = list(files_list)
        self.transform = transform
        self.labels = []
        for f in self.files:
            name = os.path.basename(f)
            first = name[0].upper() if len(name) > 0 else ""
            if first == "F":
                self.labels.append(1)
            elif first == "M":
                self.labels.append(0)
            else:
                base = name.split("_")[0].upper() if "_" in name else first
                if base == "F":
                    self.labels.append(1)
                elif base == "M":
                    self.labels.append(0)
                else:
                    raise ValueError(f"Nome de arquivo não tem F/M na frente: {name}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path)
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

Criando os tensores - data augmentation sendo feito nesta etapa

obs: a normalização está sendo feita com os pesos dos modelos pré treinados, em detrimento das estatísticas dos meus dados.

In [9]:
train_tf = T.Compose([

    T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    T.Grayscale(num_output_channels=3),

    T.RandomApply([T.ColorJitter(brightness=0.1, contrast=0.1)], p=0.2),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))], p=0.1),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=(-3, 3)),
    T.RandomResizedCrop(size=(IMAGE_HEIGHT, IMAGE_WIDTH), scale=(0.95, 1.0)),
    T.RandomAffine(degrees=0, translate=(0.03, 0.03)),

    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

    ])

val_tf = T.Compose([

    T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    T.Grayscale(num_output_channels=3),

    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

])

Split treino/validação (80/20)

In [10]:
labels_all = []
for f in all_files:
    name = os.path.basename(f)
    first = name[0].upper() if len(name) > 0 else ""
    if first == "F":
        labels_all.append(1)
    elif first == "M":
        labels_all.append(0)
    else:
        base = name.split("_")[0].upper() if "_" in name else first
        labels_all.append(1 if base == "F" else 0)

labels_all = np.array(labels_all)

train_idx, val_idx = train_test_split(
    np.arange(len(all_files)),
    test_size=0.2,
    stratify=labels_all,
    random_state=42
)
train_files = [all_files[i] for i in train_idx]
val_files   = [all_files[i] for i in val_idx]

train_dataset = FingerprintDataset(train_files, transform=train_tf)
val_dataset   = FingerprintDataset(val_files, transform=val_tf)

Aplicando pesos para a classe com menor frequência (impressões digitais de mulheres)

In [11]:
train_labels = np.array(train_dataset.labels)
class_counts = np.bincount(train_labels)

class_weights = 1.0 / (class_counts)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights.tolist(),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

Definindo modelos: Ao invés de montar uma arquitetura própria, trouxe algumas sugestões de arquiteturas pré-treinadas para classificação de imagens

In [12]:
# ---- BLOCO 11 (REESCRITO) ----

def create_model(trial):
    model_name = trial.suggest_categorical(
        "model_type",
        ["resnet18", "resnet50", "mobilenet_v2", "efficientnet_b0", "densenet121"]
    )

    dropout = trial.suggest_float("dropout", 0.0, 0.6)
    hidden_size = trial.suggest_int("hidden_size", 64, 512, log=True)
    freeze_backbone = trial.suggest_categorical("freeze_backbone", [True, False])

    # ----- INSTANCIA MODELO -----
    if model_name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    elif model_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V1")
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights="IMAGENET1K_V1")
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights="IMAGENET1K_V1")
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    elif model_name == "densenet121":
        model = models.densenet121(weights="IMAGENET1K_V1")
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    # ----- FREEZE BACKBONE -----
    if freeze_backbone:
        for name, param in model.named_parameters():
            if "fc" not in name and "classifier" not in name:
                param.requires_grad = False

    return model.to(device)


Treino e avaliação dos modelos

In [13]:
# ---- BLOCO 12 (REESCRITO) ----

def train_and_evaluate(model, trial=None, save_path=None):

    # ---------- HPS ----------
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True) if trial else 1e-4
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True) if trial else 0.0

    opt_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]) if trial else "Adam"
    momentum = trial.suggest_float("momentum", 0.5, 0.95) if (trial and opt_name == "SGD") else 0.9

    batch_size = trial.suggest_int("batch_size", 16, 64, log=True) if trial else BATCH_SIZE
    patience = trial.suggest_int("patience", 2, 10) if trial else 3
    max_epochs = trial.suggest_int("max_epochs", 10, 30) if trial else N_EPOCHS

    # ---------- RECRIA LOADER COM O BATCH SIZE ----------
    train_loader_local = DataLoader(
        train_dataset, batch_size=batch_size, sampler=sampler,
        num_workers=0, pin_memory=False
    )
    val_loader_local = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=False
    )

    # ---------- OTIMIZADORES ----------
    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:  # SGD
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0.0
    best_state = None
    no_improve = 0

    # ---------------- TREINAMENTO ----------------
    for epoch in range(max_epochs):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader_local:
            imgs = imgs.to(device)
            labels = labels.float().to(device)

            optimizer.zero_grad()
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)

        # --------------- VALIDAÇÃO ---------------
        model.eval()
        preds = []
        trues = []

        with torch.no_grad():
            for imgs, labels in val_loader_local:
                imgs = imgs.to(device)
                labels = labels.to(device)
                logits = model(imgs).squeeze(1)
                probs = torch.sigmoid(logits)
                pred_bin = (probs > 0.5).long().cpu().numpy()
                preds.extend(pred_bin.tolist())
                trues.extend(labels.cpu().numpy().tolist())

        f1 = f1_score(trues, preds, zero_division=0)

        print(f"[Epoch {epoch+1}/{max_epochs}] loss={running_loss/len(train_dataset):.4f} F1={f1:.4f}")

        # ----- EARLY STOPPING -----
        if f1 > best_f1:
            best_f1 = f1
            no_improve = 0
            best_state = model.state_dict()
            if save_path:
                torch.save(best_state, save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping.")
                break

        # Optuna prune
        if trial:
            trial.report(best_f1, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    return best_f1, best_state


Função objetivo do optuna

In [14]:
def objective(trial):
    model = create_model(trial)
    f1, _ = train_and_evaluate(model, trial=trial, save_path=None)
    return f1

Rodar optuna

In [15]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, n_jobs=-1, show_progress_bar=True)

print("Melhor trial:", study.best_trial.params)
print("Melhor F1 obtido (val):", study.best_value)

[I 2025-12-02 15:05:25,848] A new study created in memory with name: no-name-ab32fe2f-e794-4c04-93c3-2412862811b9


  0%|          | 0/30 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth



  0%|          | 0.00/30.8M [00:00<?, ?B/s]

  0%|          | 0.00/30.8M [00:00<?, ?B/s]
 38%|███▊      | 11.9M/30.8M [00:00<00:00, 124MB/s]

 15%|█▍        | 4.50M/30.8M [00:00<00:00, 46.8MB/s]
 83%|████████▎ | 25.5M/30.8M [00:00<00:00, 135MB/s]

100%|██████████| 30.8M/30.8M [00:00<00:00, 132MB/s]


100%|██████████| 30.8M/30.8M [00:00<00:00, 89.9MB/s]


[Epoch 1/26] loss=0.6861 F1=0.3694
[Epoch 1/20] loss=0.6197 F1=0.4684
[Epoch 2/26] loss=0.6710 F1=0.3962
[Epoch 2/20] loss=0.5295 F1=0.4917
[Epoch 3/26] loss=0.6470 F1=0.3950
[Epoch 3/20] loss=0.4574 F1=0.5404
[Epoch 4/26] loss=0.6310 F1=0.3934
[Epoch 4/20] loss=0.3635 F1=0.5164
[Epoch 5/26] loss=0.6420 F1=0.3984
[Epoch 5/20] loss=0.3192 F1=0.5600
[Epoch 6/26] loss=0.6271 F1=0.4016
[Epoch 6/20] loss=0.2368 F1=0.5379
[Epoch 7/26] loss=0.6336 F1=0.4144
[Epoch 7/20] loss=0.1871 F1=0.4850
Early stopping.
[I 2025-12-02 15:16:26,204] Trial 0 finished with value: 0.56 and parameters: {'model_type': 'densenet121', 'dropout': 0.06418770724342579, 'hidden_size': 244, 'freeze_backbone': False, 'lr': 5.25282253160878e-05, 'weight_decay': 0.00042947867603021603, 'optimizer': 'Adam', 'batch_size': 59, 'patience': 2, 'max_epochs': 20}. Best is trial 0 with value: 0.56.
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


...

[Epoch 1/10] loss=0.6810 F1=0.3390
[Epoch 7/14] loss=0.2532 F1=0.4800
[Epoch 2/10] loss=0.6574 F1=0.3787
[Epoch 8/14] loss=0.1974 F1=0.4751
[Epoch 3/10] loss=0.6464 F1=0.4161
[Epoch 9/14] loss=0.1690 F1=0.5723
[Epoch 4/10] loss=0.6302 F1=0.4301
[Epoch 5/10] loss=0.6319 F1=0.3993
[Epoch 10/14] loss=0.1405 F1=0.5371
[Epoch 6/10] loss=0.6228 F1=0.4417
[Epoch 11/14] loss=0.1308 F1=0.5217
[Epoch 7/10] loss=0.6168 F1=0.4348
[Epoch 12/14] loss=0.0972 F1=0.5478
[Epoch 8/10] loss=0.6137 F1=0.4250
Early stopping.
[I 2025-12-02 15:47:51,340] Trial 4 finished with value: 0.44166666666666665 and parameters: {'model_type': 'mobilenet_v2', 'dropout': 0.061895087416211414, 'hidden_size': 114, 'freeze_backbone': True, 'lr': 2.2799598268504053e-05, 'weight_decay': 4.989368326746223e-06, 'optimizer': 'Adam', 'batch_size': 28, 'patience': 2, 'max_epochs': 10}. Best is trial 0 with value: 0.56.
[Epoch 13/14] loss=0.0908 F1=0.5526
[Epoch 1/20] loss=0.6979 F1=0.2920
[Epoch 2/20] loss=0.6922 F1=0.2679
[Epoch 


  0%|          | 0.00/20.5M [00:00<?, ?B/s]
100%|██████████| 20.5M/20.5M [00:00<00:00, 108MB/s] 


[Epoch 1/12] loss=0.6235 F1=0.3871
[I 2025-12-02 15:51:43,899] Trial 6 pruned. 
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth



  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 13%|█▎        | 5.75M/44.7M [00:00<00:00, 60.1MB/s]
 26%|██▌       | 11.5M/44.7M [00:00<00:00, 58.4MB/s]
 40%|████      | 17.9M/44.7M [00:00<00:00, 61.7MB/s]
 57%|█████▋    | 25.5M/44.7M [00:00<00:00, 68.5MB/s]
100%|██████████| 44.7M/44.7M [00:00<00:00, 81.3MB/s]


[Epoch 1/23] loss=0.6591 F1=0.3764
[I 2025-12-02 15:52:11,595] Trial 7 pruned. 
[Epoch 1/30] loss=0.6956 F1=0.3340
[I 2025-12-02 15:52:37,101] Trial 8 pruned. 
[Epoch 1/26] loss=0.6974 F1=0.4299
[I 2025-12-02 15:53:35,159] Trial 9 pruned. 
[Epoch 1/20] loss=0.6925 F1=0.3660
[I 2025-12-02 15:53:47,116] Trial 10 pruned. 
[Epoch 1/16] loss=0.7151 F1=0.3955
[I 2025-12-02 15:54:53,262] Trial 11 pruned. 
[Epoch 1/16] loss=0.6794 F1=0.3906
[I 2025-12-02 15:55:10,980] Trial 12 pruned. 
[Epoch 1/16] loss=0.6421 F1=0.4322
[I 2025-12-02 15:56:15,839] Trial 13 pruned. 
[Epoch 1/15] loss=0.6322 F1=0.4778
[Epoch 1/14] loss=0.6757 F1=0.4085
[I 2025-12-02 15:57:37,416] Trial 15 pruned. 
[Epoch 2/15] loss=0.5427 F1=0.4943
[Epoch 1/18] loss=0.6967 F1=0.2028
[I 2025-12-02 15:58:59,088] Trial 16 pruned. 
[Epoch 3/15] loss=0.4696 F1=0.5552
[Epoch 1/24] loss=0.6551 F1=0.4289
[I 2025-12-02 16:00:46,042] Trial 17 pruned. 
[Epoch 4/15] loss=0.3788 F1=0.5073
[Epoch 1/14] loss=0.6334 F1=0.4118
[I 2025-12-02 16:0

Coletando melhor modelo do estudo de hiperparâmetros

In [18]:
class Dummy:
    def __init__(self, params):
        self.params = params

    def suggest_float(self, name, *args, **kwargs):
        return self.params[name]

    def suggest_categorical(self, name, choices):
        return self.params[name]

    def suggest_int(self, name, *args, **kwargs):
        return self.params[name]

    def report(self, value, step):
        pass

    def should_prune(self):
        return False


Avaliação "final"

In [20]:

if 'best_state' in globals() and best_state is not None:
    state_dict = best_state
elif 'best_model_path' in globals() and os.path.exists(best_model_path):
    state_dict = torch.load(best_model_path, map_location=device)
else:
    raise RuntimeError(
        "Nenhum 'best_state' em memória e arquivo 'best_model_path' não encontrado.\n"
        "Rode o treino final novamente e assegure que `best_state` ou `best_model_path` existam."
    )

final_model.to(device)
from collections import OrderedDict
new_state = OrderedDict()
for k, v in state_dict.items():
    new_key = k.replace("module.", "") if k.startswith("module.") else k
    new_state[new_key] = v

final_model.load_state_dict(new_state)
final_model.eval()

if 'best_params' in globals() and isinstance(best_params, dict) and 'batch_size' in best_params:
    eval_bs = int(best_params['batch_size'])
else:
    eval_bs = BATCH_SIZE

val_loader_eval = DataLoader(val_dataset, batch_size=eval_bs, shuffle=False, num_workers=0, pin_memory=False)

preds = []
trues = []
with torch.no_grad():
    for imgs, labels in val_loader_eval:
        imgs = imgs.to(device)
        labels = labels.to(device)
        logits = final_model(imgs).squeeze(1)
        probs = torch.sigmoid(logits)
        pred_bin = (probs > 0.5).long().cpu().numpy()
        preds.extend(pred_bin.tolist())
        trues.extend(labels.cpu().numpy().tolist())

print("F1 Score:", f1_score(trues, preds, zero_division=0))

Precision: 0.4024390243902439
Recall: 0.668918918918919
F1 Score: 0.5025380710659898
Confusion Matrix:
 [[505 147]
 [ 49  99]]
